# Perturbation-distance analysis of finetuning results

Relates **empirical** perturbation geometry in PCA space to **model** fit/forget scores from `metrics.json`.

| Question | x-axis | y-axis | Strata |
|----------|--------|--------|--------|
| 1. Fitting vs control distance | L2 distance between PCA means (pert vs PBS) | Finetune eval on held-out donor | Held-out context only |
| 2. Forgetting vs control distance | Same (train-phase cells) | Δ(finetune − pretrain) | Seen donors; **one point per perturbation** (median over donors) |
| 3. Fit/forget vs train↔finetune shift | L2 distance between PCA means (train vs finetune phase, per cytokine) | Fitting or forgetting metrics | Held-out / seen |
| 4. Best method on large shifts | Tertiles of train↔finetune mean shift | Median metric per method | As above |

**Proxies** use `dist_metrics.mean_l2_distance` on `obsm['X_pca']` (L2 norm of difference between group means). Up to `MAX_CELLS_PER_STRATUM` cells per stratum are loaded for tractability.

**Note:** `retrieval_mae` is one value per donor (repeated on every stratum row); per-perturbation scatter uses `pearson_delta` and PCA `energy`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO = Path("../..").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from experiments.tx.analysis.perturbation_proxies import (
    CampaignConfig,
    aggregate_by_perturbation,
    audit_campaign_analysis,
    binned_median_curve,
    default_campaign,
    high_shift_perturbations,
    load_campaign_methods_table,
    load_proxies_cached,
    merge_proxies_and_metrics,
    method_comparison_by_shift_bin,
    plot_method_shift_heatmap,
    plot_stratum_scatter,
    print_audit_report,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# --- Campaign config (switch to donor8=False for ho_d2 / Donor2) ---
import os

USE_DONOR8 = True
campaign = default_campaign(donor8=USE_DONOR8)

# Subset methods for faster iteration (None = all runs in manifest)
METHOD_FILTER = None  # e.g. ["vf0_pert0_ctx0", "vf1_pert0_ctx0", "generative_field_replay_r10", "generative_replay_r10"]

MAX_CELLS_PER_STRATUM = 256
campaign = CampaignConfig(
    bucket=campaign.bucket,
    split_col=campaign.split_col,
    held_out_context=campaign.held_out_context,
    manifest_path=campaign.manifest_path,
    max_cells_per_stratum=MAX_CELLS_PER_STRATUM,
)
os.environ["TX_RESULTS_BUCKET"] = campaign.bucket

OUT_DIR = REPO / "results_dir" / campaign.bucket / "comparisons" / "perturbation_distance_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Bucket:", campaign.bucket)
print("Held-out:", campaign.held_out_context)
print("Manifest:", campaign.manifest_path.name)
print("Output:", OUT_DIR)

In [ ]:
# Empirical distances (cached CSV; set REBUILD_PROXIES=True after proxy definition changes)
REBUILD_PROXIES = True
proxies_cache = OUT_DIR / "empirical_proxies.csv"
proxies = load_proxies_cached(campaign, proxies_cache, force=REBUILD_PROXIES)
print(f"Proxies: {len(proxies)} strata rows")
print(
    "finetune control dist (held-out donor):",
    proxies.loc[proxies["is_held_out_context"], "dist_from_control_finetune"].notna().sum(),
    "non-null",
)
proxies.head()

In [ ]:
methods = load_campaign_methods_table(campaign, method_labels=METHOD_FILTER)
print("Example labels:", sorted(methods["method"].unique())[:5], "...")

df = merge_proxies_and_metrics(proxies, methods, control_distance_col="dist_from_control_train")
df_fitting = df[df["is_held_out_context"]].copy()
df_forget = df[~df["is_held_out_context"]].copy()

print_audit_report(audit_campaign_analysis(campaign, proxies, methods))
print("Fitting strata:", len(df_fitting), "| Forgetting strata:", len(df_forget))

## 1. Fitting vs distance from control (held-out donor)

For held-out context, x-axis uses **finetune-phase** empirical distance (pert vs PBS on Donor8 cells) — matches where we evaluate fitting.

In [ ]:
df_fit_ho = merge_proxies_and_metrics(
    proxies, methods, control_distance_col="dist_from_control_finetune"
)
df_fit_ho = df_fit_ho[df_fit_ho["is_held_out_context"]].copy()
print("Q1 proxy join:", df_fit_ho["dist_from_control"].notna().mean() * 100, "% non-null")

# Manifest labels: field_replay_r10, replay_r10, vf1_pert0_ctx0, ...
REFERENCE_METHOD = "field_replay_r10"
if REFERENCE_METHOD not in df_fit_ho["method"].unique():
    print("Unknown REFERENCE_METHOD; using first available:", df_fit_ho["method"].iloc[0])
methods_to_plot = (
    [REFERENCE_METHOD]
    if REFERENCE_METHOD in df_fit_ho["method"].unique()
    else [df_fit_ho["method"].iloc[0]]
)

fit_metrics = [
    ("fitting_pearson_delta", "VCB pearson_delta (finetune)"),
    ("fitting_pca_energy", "PCA energy pred vs true (finetune)"),
]

for method in methods_to_plot:
    sub = df_fit_ho[df_fit_ho["method"] == method]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, (col, ylab) in zip(axes, fit_metrics):
        plot_stratum_scatter(
            sub,
            "dist_from_control",
            col,
            ax=ax,
            title=f"{method} — fitting",
            xlabel="L2 distance between PCA means (pert vs PBS, finetune phase)",
            ylabel=ylab,
        )
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"q1_fitting_vs_control_{method}.png", bbox_inches="tight")
    plt.show()

## 2. Forgetting vs distance from control (seen donors)

Positive Δ = worse retention after finetune (same sign convention as `summarize_fit_forget.py`).

Each point is one **perturbation**; x and y are medians over seen-donor strata (not one point per donor×perturbation).

In [ ]:
forget_metrics = [
    ("forget_pearson_delta", "Δ VCB pearson_delta"),
    ("forget_pca_energy", "Δ PCA energy"),
]

for method in methods_to_plot:
    sub = df_forget[df_forget["method"] == method]
    sub_p = aggregate_by_perturbation(sub, [c for c, _ in forget_metrics])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, (col, ylab) in zip(axes, forget_metrics):
        plot_stratum_scatter(
            sub_p,
            "dist_from_control",
            col,
            ax=ax,
            title=f"{method} — forgetting (median over seen donors)",
            xlabel="L2 distance between PCA means (pert vs PBS, train phase)",
            ylabel=ylab,
        )
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"q2_forget_vs_control_{method}.png", bbox_inches="tight")
    plt.show()

## 3. Fit / forget vs train↔finetune perturbation shift

x-axis: L2 distance between **train-phase** (multi-donor) and **finetune-phase** (held-out donor) PCA means for each cytokine.

In [ ]:
df_shift = merge_proxies_and_metrics(proxies, methods, control_distance_col="dist_from_control_train")

for method in methods_to_plot:
    sub_fit = df_shift[(df_shift["method"] == method) & df_shift["is_held_out_context"]]
    sub_fg = df_shift[(df_shift["method"] == method) & ~df_shift["is_held_out_context"]]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    panels = [
        (sub_fit, "fitting_pearson_delta", "Fitting pearson_delta"),
        (sub_fit, "fitting_pca_energy", "Fitting PCA energy"),
        (sub_fg, "forget_pearson_delta", "Δ pearson_delta (seen)"),
        (sub_fg, "forget_pca_energy", "Δ PCA energy (seen)"),
    ]
    for ax, (sub, col, title) in zip(axes.ravel(), panels):
        plot_stratum_scatter(
            sub,
            "dist_train_vs_finetune",
            col,
            ax=ax,
            title=f"{method}: {title}",
            xlabel="L2 distance between PCA means (train vs finetune phase, per cytokine)",
            ylabel=col,
        )
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"q3_shift_vs_fit_forget_{method}.png", bbox_inches="tight")
    plt.show()

## 4. Method comparison on high train↔finetune shift

Tertiles of `dist_train_vs_finetune` (pooled over strata); lower median `fitting_pca_energy` / `forget_*` is better (check sign for deltas).

In [ ]:
for subset, tag in [("fitting", "held_out"), ("forgetting", "seen")]:
    for metric, mlabel in [
        ("fitting_pearson_delta", "fitting pearson_delta"),
        ("fitting_pca_energy", "fitting PCA energy"),
        ("forget_pearson_delta", "forget pearson_delta"),
        ("forget_pca_energy", "forget PCA energy"),
    ]:
        if subset == "fitting" and not metric.startswith("fitting"):
            continue
        if subset == "forgetting" and not metric.startswith("forget"):
            continue
        summ = method_comparison_by_shift_bin(
            df_shift, metric_col=metric, subset=subset, n_bins=3
        )
        summ.to_csv(OUT_DIR / f"q4_{tag}_{metric}_by_shift.csv", index=False)
        fig = plot_method_shift_heatmap(
            summ,
            title=f"{mlabel} by train↔finetune shift tertile ({tag})",
            cmap="magma",
        )
        if fig is not None:
            fig.savefig(OUT_DIR / f"q4_{tag}_{metric}_heatmap.png", bbox_inches="tight")
            plt.show()
        else:
            print("empty summary:", tag, metric)

In [ ]:
# Rank methods within the top shift tertile only (Q4 summary table)
high_shift_p = high_shift_perturbations(df_shift, top_fraction=1 / 3)
print(f"High-shift perturbations (top tertile): n={len(high_shift_p)}")

rows = []
for method in sorted(df_shift["method"].unique()):
    sub_ho = df_shift[
        (df_shift["method"] == method)
        & df_shift["is_held_out_context"]
        & df_shift["p"].isin(high_shift_p)
    ]
    sub_seen = df_shift[
        (df_shift["method"] == method)
        & ~df_shift["is_held_out_context"]
        & df_shift["p"].isin(high_shift_p)
    ]
    rows.append(
        {
            "method": method,
            "fit_pearson_delta": sub_ho["fitting_pearson_delta"].median(),
            "fit_pca_energy": sub_ho["fitting_pca_energy"].median(),
            "forget_pearson_delta": sub_seen["forget_pearson_delta"].median(),
            "forget_pca_energy": sub_seen["forget_pca_energy"].median(),
        }
    )
rank_tbl = pd.DataFrame(rows)
display(rank_tbl.sort_values("fit_pearson_delta", ascending=False))
rank_tbl.to_csv(OUT_DIR / "q4_high_shift_method_medians.csv", index=False)

## Optional: overlay several methods (Q1 & Q2)

Use binned medians per method to compare trends without overcrowding the scatter. Q2 uses one point per perturbation (median over seen donors).

In [ ]:
COMPARE_METHODS = [
    "vf0_pert0_ctx0",
    "vf1_pert0_ctx0",
    "field_replay_r10",
    "replay_r10",
]
COMPARE_METHODS = [m for m in COMPARE_METHODS if m in df_fit_ho["method"].unique()]
print("Overlay methods:", COMPARE_METHODS)

binned_metrics = [
    ("fitting_pearson_delta", "Fitting pearson_delta"),
    ("fitting_pca_energy", "PCA energy pred vs true (finetune)"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (y_col, ylab) in zip(axes, binned_metrics):
    for method in COMPARE_METHODS:
        sub = df_fit_ho[df_fit_ho["method"] == method]
        curve = binned_median_curve(sub, "dist_from_control", y_col, n_bins=6)
        if len(curve):
            ax.plot(curve["x_mid"], curve["y_median"], marker="o", label=method)
    ax.set_xlabel("L2 distance between PCA means (pert vs PBS, finetune phase)")
    ax.set_ylabel(ylab)
    ax.set_title(f"Held-out {campaign.held_out_context}: binned trends")
axes[0].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "q1_multi_method_binned.png", bbox_inches="tight")
plt.show()

In [ ]:
COMPARE_METHODS_Q2 = [m for m in COMPARE_METHODS if m in df_forget["method"].unique()]
print("Q2 overlay methods:", COMPARE_METHODS_Q2)

forget_binned_metrics = [
    ("forget_pearson_delta", "Δ VCB pearson_delta"),
    ("forget_pca_energy", "Δ PCA energy"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (y_col, ylab) in zip(axes, forget_binned_metrics):
    for method in COMPARE_METHODS_Q2:
        sub = df_forget[df_forget["method"] == method]
        sub_p = aggregate_by_perturbation(sub, [y_col])
        curve = binned_median_curve(sub_p, "dist_from_control", y_col, n_bins=6)
        if len(curve):
            ax.plot(curve["x_mid"], curve["y_median"], marker="o", label=method)
    ax.set_xlabel("L2 distance between PCA means (pert vs PBS, train phase)")
    ax.set_ylabel(ylab)
    ax.set_title(f"Seen donors: binned forgetting trends (median over donors)")
axes[0].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "q2_multi_method_binned.png", bbox_inches="tight")
plt.show()